In [ ]:
K_db = np.zeros(N, dtype=np.float32)
A_db = np.zeros(N, dtype=np.float32)

#Calculation of mass exchange area over all bubbles in the cell n
def calc_A_db(epsilon_b_profile, y, d_b_profile_arr):
    A_db = 6 * epsilon_b_profile * A_c_bed(y) * dz / d_b_profile_arr
    return A_db


#Calculation of mass transfer coefficient between the bubble and suspension phase
def calc_K_db(u_d_profile):
    K_db = 2.7 * u_d_profile / 4
    return K_db


# Initialize storage arrays
diff_d_values_all = [[] for _ in range(N)]
diff_b_values_all = [[] for _ in range(N)]
iterations_per_cell_bed = np.zeros(N, dtype=np.int32)
density_d_all_cells = np.zeros(N, dtype=np.float32)  # Emulsion densities
density_b_all_cells = np.zeros(N, dtype=np.float32)  # Bubble densities
Y_d_all_cells = np.zeros((N, len(emulsion.species_names)), dtype=np.float32)  # Emulsion mass fractions
Y_b_all_cells = np.zeros((N, len(bubble.species_names)), dtype=np.float32)  # Bubble mass fractions
X_d_all_cells = np.zeros((N, len(emulsion.species_names)), dtype=np.float32)  # Emulsion mole fractions
X_b_all_cells = np.zeros((N, len(bubble.species_names)), dtype=np.float32)    # Bubble mole fractions
solution_d_all_cells = []
solution_b_all_cells = []
solution_C_all_cells = []
M_d_final = np.zeros((N, len(emulsion.species_names)), dtype=np.float32)
M_b_final = np.zeros((N, len(bubble.species_names)), dtype=np.float32)
converged_cells = [False] * N  # Track convergence status for each cell
mass_in_per_iter = []
mass_out_per_iter = []
mass_out_clipped_per_iter = []
rel_mass_diff_per_cell = []
C_diff_profile = []
H_diff_profile = []
O_diff_profile = []
abs_C_diff_profile = []
abs_H_diff_profile = []
abs_O_diff_profile = []


T_gasification_C = 713 # Gasification temperature (from Lagner paper, 100% SRF )
T_gasification_K = T_gasification_C + 273.15

non_C_species = [s for s in emulsion.species_names if s != "C"]


#Residence time for gases in the bed (expression from Kunii and Levenspiel p.147/271)
tau_bed_gas = 0.62 * z_bed_end / u_0 #the 0.62 comes as an approximation of the epsilon_f value
#which would be calcualted based on epsilon_f = (1-epsilon_b) * epsilon_d + epsilon_b
tau_cell_gas = tau_bed_gas/N


# Time span for ODE integration
t_span_gas = (0, tau_cell_gas) # Start and end times for gases integration
t_eval_gas = np.linspace(0, tau_cell_gas, 200)  # Time points to evaluate the solution





for n in range(N): # Loop over all cells
    log_debug(f"➡ Solving cell n = {n}")

    # Set up convergence criteria for this cell
    tolerance = 1e-5
    max_inner_iterations = 6000
    converged = False
    y_n = y[n]


    # Initialize Y_prev for this cell
    emulsion_Y_prev = emulsion.Y.copy()
    bubble_Y_prev = bubble.Y.copy()

    for iteration in range(max_inner_iterations):  # Convergence loop

        # Set gas state for this cell based on updated Y
        emulsion.TPY = T_gasification_K, P, emulsion_Y_prev
        bubble.TPY   = T_gasification_K, P, bubble_Y_prev

        # Store current values
        density_d_all_cells[n] = emulsion.density_mass
        density_b_all_cells[n] = bubble.density_mass

        Y_d_all_cells[n, :] = emulsion.Y #Store the MASS fractions
        Y_b_all_cells[n, :] = bubble.Y

        #Commenting it so it runs faster
        #log_debug(f"Initial Y_d: {Y_d_all_cells[n, :]}")
        #log_debug(f"Initial Y_b: {Y_b_all_cells[n, :]}")

        X_d_all_cells[n, :] = emulsion.X #Store the MOLE fractions
        X_b_all_cells[n, :] = bubble.X
        


        #Update fluidization properties with current d_b
        u_mf_i = calc_umf(g, d_sv, rho_p, emulsion, sphericity, epsilon_mf)

        #THIS IS THE EXTRA THING I PUT FOR THE SECOND FORMULA
        d_bubble_i = calc_bubble_growth(y_n, u_0, g, d_sv, rho_p, emulsion, sphericity, epsilon_mf)

        # THESE 3 ARE GOOD - I KEEP THEM NO MATTER HOW I CALCULATE D_BUBBLE
        psi_b_i = calc_psib(g, d_sv, rho_p, emulsion, sphericity, epsilon_mf)
        u_d_i = calc_ud(g, d_sv, rho_p, emulsion, sphericity, epsilon_mf, u_0)
        eps_d_i = calc_epsilon_d(g, d_sv, rho_p, emulsion, sphericity, d_p, epsilon_mf, u_0)
        u_b_i = solve_u_b(y_n, d_bubble_i, eps_d_i, emulsion, u_0, g, d_p, d_sv, rho_p, sphericity, epsilon_mf)[0] #bubble velocity ub
        u_b_i_profile = solve_u_b(y_n, d_bubble_i, eps_d_i, emulsion, u_0, g, d_p, d_sv, rho_p, sphericity, epsilon_mf)[1] #bubble rise velocity u_b_i
        eps_b_i = calc_epsilon_b(y_n, g, d_sv, rho_p, emulsion, sphericity, epsilon_mf, d_p, u_0, n_b, d_bubble_i)
        Ar_i = calc_Ar(g, d_sv, rho_p, emulsion)


        # Save current u_b and epsilon_b for plotting or further use
        u_b_profile_arr[n] = u_b_i
        u_b_i_profile_arr[n] = u_b_i_profile
        epsilon_b_profile_arr[n] = eps_b_i
        epsilon_d_profile_arr[n] = eps_d_i
        u_mf_profile_arr[n] = u_mf_i
        u_d_profile_arr[n] = u_d_i
        Ar_arr[n] = Ar_i
        psi_b[n] = psi_b_i
        d_b_profile_arr[n] = d_bubble_i

        A_db_n =  calc_A_db(epsilon_b_profile_arr[n], y[n], d_b_profile_arr[n])
        K_db_n =  calc_K_db(u_d_profile_arr[n])
        V_cell_n = V_cell(y[n], dz)  # Compute V_cell for this cell


        # Mass inflows per cell
        m_in_b_n = m_in_b[n, :]  # Take actual mass inflow values

        #Initial mass values for emulsion and bubble phases
        if n == 0: # Your current biomass + pyrolysis yield logic
            M_d_0 = np.zeros(len(emulsion.species_names))
            M_d_0 = np.array([biomass_flow_feed * tau_cell_gas * Y_pyrolysis[sp] for sp in emulsion.species_names])
            M_d_0[emulsion.species_index("C")] = 0.0

            M_C_0 = np.array([biomass_flow_feed * tau_cell_gas *  Y_pyrolysis["C"]])

            M_b_0 = np.zeros(len(bubble.species_names))
            M_b_0 = np.array([total_co2_flow * tau_cell_gas * Y_init[sp] for sp in bubble.species_names])# kg of CO2 in cell 0 over residence time

            # Compute total inflow for this iteration (same each time for n == 0)
            mass_in_now = np.sum(M_d_0) + np.sum(M_C_0) + np.sum(M_b_0)

            # Save to list
            mass_in_per_iter.append(mass_in_now)

        else:
            M_d_0 = M_d_final[n - 1, :].copy()
            idx_C = emulsion.species_index("C")
            M_d_0[idx_C] = 0.0  # Zero the carbon mass, as it is handled separately
            if np.sum(M_d_0) < 1e-8:
                log_debug(f"⚠ Cell {n}: Very low mass inflow detected, injecting minimum synthetic mass.")
                for i, species in enumerate(emulsion.species_names):
                    if species != "C":
                      M_d_0[i] = 1e-9  # Or distribute total 1e-8 over species

            M_b_0 = M_b_final[n - 1, :].copy()

            M_C_0 = np.array([M_d_final[n - 1, emulsion.species_index("C")]])


        # Solve ODEs
        #πριν ειχα μεθοδο BDF, τωρα δοκιμαζω LSODA μπασ και ειναι πιο γρηγορο
        sol_d_obj = solve_ivp(dMd_dt, t_span_gas, M_d_0, method='LSODA', t_eval=t_eval_gas,
                      args=(emulsion, bubble, K_db_n, A_db_n, V_cell_n, n, iteration, density_d_all_cells, Y_d_all_cells, u_d_profile_arr, epsilon_d_profile_arr, epsilon_b_profile_arr), rtol=1e-6, atol=1e-9)

        sol_C_obj = solve_ivp(dM_C_dt, t_span_gas, M_C_0, method='LSODA', t_eval=t_eval_gas,
                              args=(emulsion, V_cell_n, n, Ar_arr[n], iteration, density_d_all_cells, Y_d_all_cells, u_d_profile_arr, epsilon_d_profile_arr, Ar_arr, epsilon_b_profile_arr, u_b_i_profile_arr), rtol=1e-6, atol=1e-9)

        sol_b_obj = solve_ivp(dMb_dt, t_span_gas, M_b_0, method='LSODA', t_eval=t_eval_gas,
                      args=(emulsion, bubble, K_db_n, A_db_n, m_in_b[n, :], V_cell_n, n, iteration, density_b_all_cells, Y_b_all_cells, u_b_profile_arr, epsilon_b_profile_arr), rtol=1e-6, atol=1e-9)


        # Access the final values
        sol_d = sol_d_obj.y.T
        sol_b = sol_b_obj.y.T
        sol_C = sol_C_obj.y.T


        M_d_partial = np.clip(sol_d_obj.y.T[-1, :], 0, None)

        for i, species in enumerate(non_C_species):
            idx = emulsion.species_index(species)
            M_d_final[n, idx] = M_d_partial[i]

        #Commenting so it runs faster
        #log_debug(f"Initial M_d (before adding C): {M_d_final[n, :]}")

        M_C_final = np.clip(sol_C_obj.y.T[-1, 0], 0, None)
        idx_C = emulsion.species_index("C")
        M_d_final[n, idx_C] = M_C_final # add the mass of C back into M_d_final

        M_b_final[n, :] = np.clip(sol_b[-1, :], 0, None)


        # Now you can safely compute emulsion_sum / Normalize
        emulsion_sum = np.sum(M_d_final[n, :])
        bubble_sum = np.sum(M_b_final[n, :])

        if np.any(M_d_final[n, :] < 0) or emulsion_sum < 1e-10:
            log_debug(f"⚠ Invalid emulsion mass in cell {n}, iteration {iteration}. Reverting.")
            emulsion_Y_new = emulsion_Y_prev.copy()
        else:
            emulsion_Y_new = M_d_final[n, :] / emulsion_sum

        if np.any(M_b_final[n, :] < 0) or bubble_sum < 1e-10:
            log_debug(f"⚠ Invalid bubble mass in cell {n}, iteration {iteration}. Reverting.")
            bubble_Y_new = bubble_Y_prev.copy()
        else:
            bubble_Y_new = M_b_final[n, :] / bubble_sum


        # Optional: clip and re-normalize
        emulsion_Y_new = np.clip(emulsion_Y_new, 0, 1)
        bubble_Y_new = np.clip(bubble_Y_new, 0, 1)

        emulsion_Y_sum = np.sum(emulsion_Y_new)
        bubble_Y_sum = np.sum(bubble_Y_new)

        #Prevents division by zero or by a super small number (which could explode numerically).
        if emulsion_Y_sum > 1e-6:
            emulsion_Y_new /= emulsion_Y_sum
        else:
            log_debug(f"⚠ WARNING: Emulsion mass fraction sum {emulsion_Y_sum:.2e} too low at cell {n}, iteration {iteration}. Using previous Y.")
            emulsion_Y_new = emulsion_Y_prev.copy()


        if bubble_Y_sum > 1e-6:
            bubble_Y_new /= bubble_Y_sum
        else:
            log_debug(f"⚠ WARNING: Emulsion mass fraction sum {bubble_Y_sum:.2e} too low at cell {n}, iteration {iteration}. Using previous Y.")
            bubble_Y_new = bubble_Y_prev.copy()


        # Debugging: Check for NaNs in normalized mass fractions
        assert not np.any(np.isnan(emulsion_Y_new)), f"❌ NaNs in emulsion_Y_new at cell {n}, iteration {iteration}"
        assert not np.any(np.isnan(bubble_Y_new)), f"❌ NaNs in bubble_Y_new at cell {n}, iteration {iteration}"

        assert not np.any(np.isinf(emulsion_Y_new)), f"❌ Infs in emulsion_Y_new at cell {n}, iteration {iteration}"
        assert not np.any(np.isinf(bubble_Y_new)), f"❌ Infs in bubble_Y_new at cell {n}, iteration {iteration}"


        # Check convergence for this cell
        diff_d = np.max(np.abs(emulsion_Y_new - emulsion_Y_prev))
        diff_b = np.max(np.abs(bubble_Y_new - bubble_Y_prev))

        diff_d_values_all[n].append(diff_d)
        diff_b_values_all[n].append(diff_b)


        if diff_d < tolerance and diff_b < tolerance:
            log_debug(f"✅ Converged at cell {n} after {iteration + 1} iterations")
            converged = True
            converged_cells[n] = True
            iterations_per_cell_bed[n] = iteration + 1  # Store the number of iterations (1-based)
            break

        # Under-relaxation
        alpha_em = 0.0024
        alpha_b = 0.01

        if n == 0 or n==1: #higher relaxation factor to ensure convergence
          alpha_em = 0.003
          alpha_b = 0.01


        if n == 999: #higher relaxation factor to ensure convergence
          alpha_em = 0.003
          alpha_b = 0.01

        emulsion_Y_prev = (alpha_em) * emulsion_Y_new + (1 - alpha_em) * emulsion_Y_prev
        bubble_Y_prev = alpha_b * bubble_Y_new + (1 - alpha_b) * bubble_Y_prev

        #Commenting so it runs faster
        # Printing final values
        #if diff_d < tolerance and diff_b < tolerance or iteration == max_inner_iterations - 1:
        #    log_debug(f"\n📦 Cell {n} ODE results (iteration {iteration + 1}):")
        #    log_debug(f"  M_d_final: {M_d_final[n, :]}")
        #    log_debug(f"  M_b_final: {M_b_final[n, :]}")
        #    log_debug(f"  ⛽ Emulsion mass sum before normalization: {emulsion_sum:.6e}")
        #    log_debug(f"  💨 Bubble mass sum before normalization: {bubble_sum:.6e}")
        #    log_debug(f"  ✅ Emulsion mass fractions (Y_d): {emulsion_Y_new}")
        #    log_debug(f"  ✅ Bubble mass fractions (Y_b): {bubble_Y_new}")
        #    log_debug(f"  🔁 Emulsion mass fraction sum: {np.sum(emulsion_Y_new):.6f}")
        #    log_debug(f"  🔁 Bubble mass fraction sum: {np.sum(bubble_Y_new):.6f}")


    if not converged:
        log_debug(f"⚠ Cell {n} did not fully converge after {max_inner_iterations} iterations")

    mass_in_cell = np.sum(M_d_0) + np.sum(M_C_0) + np.sum(M_b_0) + np.sum(m_in_b[n, :]) #incoming mass from previous cell and nozzles
    mass_out_cell = np.sum(M_d_final[n, :]) + np.sum(M_b_final[n, :])  # includes C already
    mass_diff_cell = mass_out_cell - mass_in_cell
    rel_diff_cell = 100 * mass_diff_cell / mass_in_cell  # percent

    # Save to list
    rel_mass_diff_per_cell.append(rel_diff_cell)

    # Set final mass fractions for this cell
    Y_d_all_cells[n, :] = emulsion_Y_new
    Y_b_all_cells[n, :] = bubble_Y_new



# --------------------- PLOTS -------------------------------


# Plot how many cells converged
num_converged = sum(converged_cells)
log_debug(f"✅ {num_converged}/{N} cells converged")
print(f"✅ {num_converged}/{N} cells converged")

plt.figure()
plt.bar(["Converged", "Not Converged"], [num_converged, N - num_converged])
plt.title("Number of Cells Converged")
plt.ylabel("Number of Cells")
plt.show()


# Identify non-converged cells
non_converged_cells = [i for i, converged in enumerate(converged_cells) if not converged]
if non_converged_cells:
    print(f"⚠ The following cells did NOT converge: {non_converged_cells}")
else:
    print("✅ All cells converged!")


# Plot the relative mass difference for CELL 0 across all iterations with the UNCLIPPED values
#mass_in_arr = np.array(mass_in_per_iter)
#mass_out_arr = np.array(mass_out_per_iter)
#mass_diff_arr = mass_out_arr - mass_in_arr
#rel_diff_arr = mass_diff_arr / mass_in_arr * 100  # percentage
#plt.figure(figsize=(8, 4))
#plt.plot(rel_diff_arr, marker='o')
#plt.axhline(0, color='gray', linestyle='--', label='Perfect Balance')
#plt.ylabel('Relative Mass Difference (%)')
#plt.xlabel('Iteration')
#plt.title('Cell 0 | Mass Balance per Iteration (unclipped values)')
#plt.ylim(-5, 20)  # 👈 Zoom in on the y-axis
#plt.grid(True)
#plt.legend()
#plt.tight_layout()
#plt.show()

# Plot emulsion convergence of cell 0
#plt.figure()
#plt.plot(diff_d_values_all[0], label='Cell 0')
#plt.yscale('log')
#plt.xlabel('Inner iteration')
#plt.ylabel('Emulsion difference (diff_d)')
#plt.title('Convergence Behavior in Cell 0')
#plt.grid(True)
#plt.legend()
#plt.show()


# Plot bubble convergence of cell 0
#plt.figure()
#plt.plot(diff_b_values_all[0], label='Cell 0')
#plt.yscale('log')
#plt.xlabel('Inner iteration')
#plt.ylabel('Bubble difference (diff_d)')
#plt.title('Convergence Behavior in Cell 0')
#plt.grid(True)
#plt.legend()
#plt.show()

#Evolution of UNCLIPPED masses of emulsion species for CELL 0
#num_species_emulsion = mass_evolution_cell0_emulsion.shape[1]
#for i, species in enumerate(non_C_species):
#    plt.figure(figsize=(7, 4))
#    plt.plot(range(len(mass_evolution_cell0_emulsion)), mass_evolution_cell0_emulsion[:, i])
#    plt.xlabel('Iteration')
#    plt.ylabel('Mass (kg)')
#    plt.title(f'Evolution of {species} Mass in Emulsion Phase (Cell 0)')
#    plt.grid(True)
#    plt.tight_layout()
#    plt.show()

#Commenting so it runs faster
#Evolution of CLIPPED masses of emulsion species for CELL 0
#num_species_emulsion = mass_evolution_cell0_emulsion_CLIPPED.shape[1]
#for i, species in enumerate(emulsion.species_names):
#    plt.figure(figsize=(7, 4))
#    plt.plot(range(len(mass_evolution_cell0_emulsion_CLIPPED)), mass_evolution_cell0_emulsion_CLIPPED[:, i])
#    plt.xlabel('Iteration')
#    plt.ylabel('Mass (kg)')
#    plt.title(f'Evolution of CLIPPED {species} Mass in Emulsion Phase (Cell 0)')
#    plt.grid(True)
#    plt.tight_layout()
#    plt.show()


#Evolution of UNCLIPPED masses of bubble species for CELL 0
#num_species_bubble = mass_evolution_cell0_bubble.shape[1]
#for i, species in enumerate(non_C_species):  # Assuming bubble species match non_C_species
#    plt.figure(figsize=(7, 4))
#    plt.plot(range(len(mass_evolution_cell0_bubble)), mass_evolution_cell0_bubble[:, i])
#    plt.xlabel('Iteration')
#    plt.ylabel('Mass (kg)')
#    plt.title(f'Evolution of {species} Mass in Bubble Phase (Cell 0)')
#    plt.grid(True)
#    plt.tight_layout()
#    plt.show()



#Commenting so it runs faster
#Evolution of CLIPPED masses of bubble species for CELL 0
#num_species_bubble = mass_evolution_cell0_bubble_CLIPPED.shape[1]
#for i, species in enumerate(non_C_species):  # Assuming bubble species match non_C_species
#    plt.figure(figsize=(7, 4))
#    plt.plot(range(len(mass_evolution_cell0_bubble_CLIPPED)), mass_evolution_cell0_bubble_CLIPPED[:, i])
#    plt.xlabel('Iteration')
#    plt.ylabel('Mass (kg)')
#    plt.title(f'Evolution of CLIPPED {species} Mass in Bubble Phase (Cell 0)')
#    plt.grid(True)
#    plt.tight_layout()
#    plt.show()